In [114]:
import sympy as sp
from sympy import cos, sin, pi, Integer, sqrt
import re

In [115]:
f = sp.Matrix([sp.Symbol(f"v[{i}]") for i in range(15)])
target = sp.Matrix([sp.Symbol(f"y[{i}]") for i in range(15)])
scales = sp.Matrix([sp.Symbol(f"l_{i}") for i in ["x", "y", "z"]])
C = sp.Symbol("C")

alpha = sp.Symbol("alpha")
beta = sp.Symbol("beta")
gamma = sp.Symbol("gamma")

c, s, c2, c3, c4, s2, s3, s4 = sp.symbols("c,s,c2,c3,c4,s2,s3,s4")

In [116]:
# Matches decimal/scientific notation numbers but not integers
float_re = re.compile(
    r'(?<![\w.])'
    r'([+-]?(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?'
    r'|[+-]?\d+[eE][+-]?\d+)'
)

def wrap_constants(code):
    return code
    return float_re.sub(r'constant<T>(\1)', code)

def to_ccode_vector(expr):
    ev = expr.evalf()
    out = "{\n"

    for i in range(len(ev)):
        code = sp.ccode(sp.simplify(ev[i]))
        code = wrap_constants(code)
        out += f"    {code}"
        if i != len(ev) - 1:
            out += ",\n"

    out += "\n};"
    return out

def to_ccode_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        row = "    {"
        for j in range(ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            row += "" + code
            if j != ev.cols - 1:
                row += ","
            else:
                row += "}"
        out += row
        if i != ev.rows - 1:
            out += ",\n"
    out += "\n};"
    return out

def to_ccode_sym_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        for j in range(i, ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            out += "    " + code
            if i != ev.rows - 1:
                out += ",\n"

    out += "};"
    return out

In [117]:
rotate_z_5d = sp.Matrix([
    [c2, 0, 0, 0, s2],
    [0, c, 0, s, 0],
    [0, 0, 1, 0, 0],
    [0, -s, 0, c, 0],
    [-s2, 0, 0, 0, c2]
])

rotate_z_9d = sp.Matrix([
    [c4, 0, 0, 0, 0, 0, 0, 0, s4],
    [0, c3, 0, 0, 0, 0, 0 , s3, 0],
    [0, 0, c2, 0, 0, 0, s2, 0, 0],
    [0, 0, 0, c, 0, s, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, -s, 0, c, 0, 0, 0],
    [0, 0, -s2, 0, 0, 0, c2, 0, 0],
    [0, -s3, 0, 0, 0, 0, 0, c3, 0],
    [-s4, 0, 0, 0, 0, 0, 0, 0, c4]
])

rotate_z = sp.diag(
    sp.Integer(1),
    rotate_z_5d,
    rotate_z_9d
)

rot_z_f = rotate_z * f

print(to_ccode_vector(rot_z_f))
#print(to_ccode_matrix(rotate_z))

{
    v[0],
    c2*v[1] + s2*v[5],
    c*v[2] + s*v[4],
    v[3],
    c*v[4] - s*v[2],
    c2*v[5] - s2*v[1],
    c4*v[6] + s4*v[14],
    c3*v[7] + s3*v[13],
    c2*v[8] + s2*v[12],
    c*v[9] + s*v[11],
    v[10],
    c*v[11] - s*v[9],
    c2*v[12] - s2*v[8],
    c3*v[13] - s3*v[7],
    c4*v[14] - s4*v[6]
};


In [118]:
rot_x_pi_over_two_band_2 = sp.Matrix([
    [0, 0, 0, -1, 0],
    [0, -1, 0, 0, 0],
    [0, 0, -Integer(1)/2, 0, -sqrt(3)/2],
    [1, 0, 0, 0, 0],
    [0, 0, -sqrt(3)/2, 0, Integer(1)/2]
])

rot_x_pi_over_two_band_4 = sp.Matrix([
    [0, 0, 0, 0, 0, sqrt(14)/4, 0, - sqrt(2)/4, 0],
    [0, -Integer(3)/4, 0, sqrt(7)/4, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, sqrt(2)/4, 0, sqrt(14)/4, 0],
    [0, sqrt(7)/4, 0, Integer(3)/4, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, Integer(3)/8, 0, sqrt(5)/4, 0, sqrt(35)/8],
    [-sqrt(14)/4, 0, -sqrt(2)/4, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, sqrt(5)/4, 0, 1/2, 0, -sqrt(7)/4],
    [sqrt(2)/4, 0, -sqrt(14)/4, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, sqrt(35)/8, 0, -sqrt(7)/4, 0, Integer(1)/8]
])

rot_x_pi_2 = sp.diag(
    Integer(1),
    rot_x_pi_over_two_band_2,
    rot_x_pi_over_two_band_4
)


rotate_y = rot_x_pi_2 * rotate_z.T * rot_x_pi_2.T
rotate_y

rot_y_f = rotate_y * f
rot_y_f = sp.simplify(rot_y_f)

#print(to_ccode_matrix(rotate_y))
print(to_ccode_vector(rot_y_f))

{
    v[0],
    c*v[1] + s*v[2],
    c*v[2] - s*v[1],
    -0.8660254037844386*s2*v[4] + 0.25*v[3]*(3.0*c2 + 1.0) - 0.4330127018922193*v[5]*(c2 - 1.0),
    c2*v[4] + 0.8660254037844386*s2*v[3] - 0.5*s2*v[5],
    0.5*s2*v[4] - 0.4330127018922193*v[3]*(c2 - 1.0) + 0.25*v[5]*(c2 + 3.0),
    0.125*v[6]*(7.0*c + c3) + 0.088388347648318447*v[7]*(7.0*s + 3.0*s3) + 0.33071891388307384*v[8]*(c - c3) + 0.23385358667337133*v[9]*(3.0*s - s3),
    -0.088388347648318447*v[6]*(7.0*s + 3.0*s3) + 0.0625*v[7]*(7.0*c + 9.0*c3) - 0.23385358667337133*v[8]*(s - 3.0*s3) + 0.49607837082461076*v[9]*(c - c3),
    0.33071891388307384*v[6]*(c - c3) + 0.23385358667337133*v[7]*(s - 3.0*s3) + 0.125*v[8]*(c + 7.0*c3) + 0.088388347648318447*v[9]*(3.0*s + 7.0*s3),
    -0.23385358667337133*v[6]*(3.0*s - s3) + 0.49607837082461076*v[7]*(c - c3) - 0.088388347648318447*v[8]*(3.0*s + 7.0*s3) + 0.0625*v[9]*(9.0*c + 7.0*c3),
    0.015625*v[10]*(20.0*c2 + 35.0*c4 + 9.0) - 0.09882117688026186*v[11]*(2.0*s2 + 7.0*s4) + 0.069877124

In [119]:
subs = {
    s: 1,
    c: 0,
    s2: 0,
    c2: -1,
    s3: -1,
    c3: 0,
    s4: 0,
    c4: 1,
}

rot_y_pi_over_two = rotate_y.subs(subs)

rotate_x = rot_y_pi_over_two.T * rotate_z.T * rot_y_pi_over_two

rot_x_f = rotate_x * f

#print(to_ccode_matrix(rotate_x))
print(to_ccode_vector(rot_x_f))

{
    v[0],
    c*v[1] - s*v[4],
    c2*v[2] - 0.8660254037844386*s2*v[3] - 0.5*s2*v[5],
    0.8660254037844386*s2*v[2] + v[3]*(0.75*c2 + 0.25) + 0.4330127018922193*v[5]*(c2 - 1),
    c*v[4] + s*v[1],
    0.5*s2*v[2] + 0.4330127018922193*v[3]*(c2 - 1) + v[5]*(0.25*c2 + 0.75),
    v[11]*(0.70156076002011403*s - 0.23385358667337133*s3) - v[13]*(0.61871843353822908*s + 0.26516504294495535*s3) + v[6]*(0.875*c + 0.125*c3) - 0.33071891388307384*v[8]*(c - c3),
    v[10]*(0.52291251658379723*s2 - 0.26145625829189861*s4) - v[12]*(0.46770717334674267*s2 + 0.23385358667337133*s4) - v[14]*(0.61871843353822908*s2 + 0.044194173824159223*s4) + v[7]*(0.875*c2 + 0.125*c4) - 0.33071891388307384*v[9]*(c2 - c4),
    -v[11]*(0.26516504294495535*s + 0.61871843353822908*s3) + v[13]*(0.23385358667337133*s - 0.70156076002011403*s3) - 0.33071891388307384*v[6]*(c - c3) + v[8]*(0.125*c + 0.875*c3),
    -v[10]*(0.19764235376052372*s2 + 0.69174823816183306*s4) + v[12]*(0.17677669529663689*s2 - 0.61871843353822908*s

# Derivatives

In [120]:

L_x_5d = sp.Matrix([
    [0, 0, 0, -1, 0],
    [0, 0, -sqrt(3), 0, -1],
    [0, sqrt(3), 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0]
])

L_y_5d = sp.Matrix([
    [0, 1, 0, 0, 0],
    [-1, 0, 0, 0, 0],
    [0, 0, 0, -sqrt(3), 0],
    [0, 0, sqrt(3), 0, -1],
    [0, 0, 0, 1, 0]
])

L_z_5d = sp.Matrix([
    [0, 0, 0, 0, 2],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, -1, 0, 0, 0],
    [-2, 0, 0, 0, 0]
])

L_x_9d = sp.Matrix([
    [0, 0, 0, 0, 0, 0, 0, -sqrt(2), 0],
    [0, 0, 0, 0, 0, 0, -sqrt(Integer(7)/2), 0, -sqrt(2)],
    [0, 0, 0, 0, 0, -Integer(3)/sqrt(2), 0, -sqrt(Integer(7)/2), 0],
    [0, 0, 0, 0, -sqrt(10), 0, -Integer(3)/sqrt(2), 0, 0],
    [0, 0, 0, sqrt(10), 0, 0, 0, 0, 0],
    [0, 0, 3/sqrt(2), 0, 0, 0, 0, 0, 0],
    [0, sqrt(Integer(7)/2), 0, Integer(3)/sqrt(2), 0, 0, 0, 0, 0],
    [sqrt(2), 0, sqrt(Integer(7)/2), 0, 0, 0, 0, 0, 0],
    [0, sqrt(2), 0, 0, 0, 0, 0, 0, 0]
])

L_y_9d = sp.Matrix([
    [0, sqrt(2), 0, 0, 0, 0, 0, 0, 0],
    [-sqrt(2), 0, sqrt(Integer(7)/2), 0, 0, 0, 0, 0, 0],
    [0, -sqrt(Integer(7)/2), 0, Integer(3)/sqrt(2), 0, 0, 0, 0, 0],
    [0, 0, -Integer(3)/sqrt(2), 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, -sqrt(10), 0, 0, 0],
    [0, 0, 0, 0, sqrt(10), 0, -3/sqrt(2), 0, 0],
    [0, 0, 0, 0, 0, 3/sqrt(2), 0, -sqrt(Integer(7)/2), 0],
    [0, 0, 0, 0, 0, 0, sqrt(Integer(7)/2), 0, -sqrt(2)],
    [0, 0, 0, 0, 0, 0, 0, sqrt(2), 0]
])

L_z_9d = sp.Matrix([
    [0, 0, 0, 0, 0, 0, 0, 0, 4],
    [0, 0, 0, 0, 0, 0, 0, 3, 0],
    [0, 0, 0, 0, 0, 0, 2, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -1, 0, 0, 0, 0, 0],
    [0, 0, -2, 0, 0, 0, 0, 0, 0],
    [0, -3, 0, 0, 0, 0, 0, 0, 0],
    [-4, 0, 0, 0, 0, 0, 0, 0, 0]
])

L_x = sp.diag(
    Integer(0),
    L_x_5d,
    L_x_9d
)

L_y = sp.diag(
    Integer(0),
    L_y_5d,
    L_y_9d
)

L_z = sp.diag(
    Integer(0),
    L_z_5d,
    L_z_9d
)

F = sp.Matrix([
    [sqrt(pi) * Integer(2) / 5, sqrt(pi) * Integer(2) / 5, sqrt(pi) * Integer(2) / 5],

    [0, 0, 0],
    [0, 0, 0],
    [-(Integer(4) * sqrt(pi)) / (7 * sqrt(5)), -(4 * sqrt(pi)) / (7 * sqrt(5)), 2 * (4 * sqrt(pi)) / (7 * sqrt(5))],
    [0, 0, 0],
    [(4 * sqrt(3 * pi)) / (7 * sqrt(5)), -(4 * sqrt(3 * pi)) / (7 * sqrt(5)), 0],

    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
    [2 * sqrt(pi) / 35, 2 * sqrt(pi) / 35, 16 * sqrt(pi) / 105],
    [0, 0, 0],
    [-(4 * sqrt(pi)) / (21 * sqrt(5)), (4 * sqrt(pi)) / (21 * sqrt(5)), 0],
    [0, 0, 0],
    [(2 * sqrt(pi)) / (3 * sqrt(35)), (2 * sqrt(pi)) / (3 * sqrt(35)), 0]
])


grad = sp.Matrix.vstack(
    sp.Matrix([
        -2 * target.T * L_x * F * scales,
        -2 * target.T * L_y * F * scales,
        -2 * target.T * L_z * F * scales
    ]),
    2 * F.T * F * scales - 2 * F.T * target
)

grad[3] -= C / scales[0]
grad[4] -= C / scales[1]
grad[5] -= C / scales[2]

print(to_ccode_vector(grad))

{
    1.1102230246251565e-16*l_x*y[7] + l_y*(-3.1381413698186362*y[2] + 1.1298600273745016*y[7] + 1.2811408494623837*y[9]) + l_z*(3.1381413698186362*y[2] + 1.7081877992831784*y[9]),
    l_x*(-1.2811408494623837*y[11] + 1.1298600273745016*y[13] + 3.1381413698186362*y[4]) + 1.1102230246251565e-16*l_y*y[13] - l_z*(1.7081877992831784*y[11] + 3.1381413698186362*y[4]),
    -l_x*(3.1381413698186362*y[1] + 1.5978633742962567*y[6] - 0.60393558820663018*y[8]) - l_y*(-3.1381413698186362*y[1] + 1.5978633742962567*y[6] + 0.60393558820663018*y[8]),
    -C/l_x + 2.7925268031909272*l_x + 0.23935944027350806*l_y + 0.23935944027350806*l_z - 1.4179630807244128*y[0] - 0.20256615438920181*y[10] + 0.30196779410331509*y[12] - 0.39946584357406417*y[14] + 0.90590338230994527*y[3] - 1.5690706849093181*y[5],
    -C/l_y + 0.23935944027350806*l_x + 2.7925268031909272*l_y + 0.23935944027350806*l_z - 1.4179630807244128*y[0] - 0.20256615438920181*y[10] - 0.30196779410331509*y[12] - 0.39946584357406417*y[14] + 0.90590

In [121]:
h1 = -2 * sp.Matrix([
    [target.T * L_x * L_x * F * scales, target.T * L_x * L_y * F * scales, target.T * L_x * L_z * F * scales],
    [target.T * L_y * L_x * F * scales, target.T * L_y * L_y * F * scales, target.T * L_y * L_z * F * scales],
    [target.T * L_z * L_x * F * scales, target.T * L_z * L_y * F * scales, target.T * L_z * L_z * F * scales]
])

h2 = -2 * sp.Matrix([
    (F.T * L_x.T * target).T,
    (F.T * L_y.T * target).T,
    (F.T * L_z.T * target).T
])

h3 = h2.T

h4 = 2 * F.T * F + C * sp.Matrix([[1 / (scales[0]**2), 0, 0], [0, 1/ (scales[1]**2), 0], [0, 0, 1 / (scales[2] ** 2)]])

hess = sp.BlockMatrix([
    [h1, h2],
    [h3, h4]
])

print(to_ccode_matrix(sp.Matrix(hess)))

{
    {2.0*l_x*(2.2204460492503131e-16*y[12] + 5.5511151231257827e-17*y[14]) + 2.0*l_y*(2.0256615438920185*y[10] + 2.4157423528265207*y[12] + 0.79893168714812823*y[14] - 2.7177101469298357*y[3] - 1.5690706849093181*y[5]) + 2.0*l_z*(2.7008820585226916*y[10] + 1.8118067646198912*y[12] + 2.7177101469298357*y[3] + 1.5690706849093181*y[5]),-2.0*l_x*(1.5690706849093181*y[1] + 0.79893168714812823*y[6] - 0.3019677941033152*y[8]) - 1.1102230246251565e-16*l_y*(y[6] - y[8]) + 2.0*l_z*(1.5690706849093181*y[1] + 1.8118067646198905*y[8]),-2.0*l_x*(-0.64057042473119186*y[11] + 0.56493001368725093*y[13] + 1.5690706849093181*y[4]) - 2.0*l_y*(0.64057042473119186*y[11] + 1.6947900410617525*y[13] - 1.5690706849093181*y[4]),0,-3.1381413698186362*y[2] + 1.1298600273745016*y[7] + 1.2811408494623837*y[9],3.1381413698186362*y[2] + 1.7081877992831784*y[9]},
    {1.1102230246251565e-16*l_x*y[6] + 2.0*l_y*(-1.5690706849093181*y[1] + 0.79893168714812823*y[6] + 0.30196779410331515*y[8]) + 2.0*l_z*(1.569070684909318

# Z aligned

In [124]:
# gradient

scales = sp.Matrix([sp.Symbol("l_x"), sp.Symbol("l_y"), Integer(1)])

grad = sp.Matrix([
    -2 * target.T * L_z * F * scales
])

grad_scales_full = 2 * F.T * F * scales - 2 * F.T * target

grad = sp.Matrix([
    grad,
    grad_scales_full[0] - C * scales[0],
    grad_scales_full[1] - C * scales[1]
])

print(to_ccode_vector(grad))

{
    -l_x*(3.1381413698186362*y[1] + 1.5978633742962567*y[6] - 0.60393558820663018*y[8]) - l_y*(-3.1381413698186362*y[1] + 1.5978633742962567*y[6] + 0.60393558820663018*y[8]),
    -C*l_x + 2.7925268031909272*l_x + 0.23935944027350806*l_y - 1.4179630807244128*y[0] - 0.20256615438920181*y[10] + 0.30196779410331509*y[12] - 0.39946584357406417*y[14] + 0.90590338230994527*y[3] - 1.5690706849093181*y[5] + 0.23935944027350806,
    -C*l_y + 0.23935944027350806*l_x + 2.7925268031909272*l_y - 1.4179630807244128*y[0] - 0.20256615438920181*y[10] - 0.30196779410331509*y[12] - 0.39946584357406417*y[14] + 0.90590338230994527*y[3] + 1.5690706849093181*y[5] + 0.23935944027350806
};


In [125]:
# hessian

h00 = -2 * target.T * L_z * L_z * F * scales
h00 = h00[0, 0]

hess_theta_full = -2 * F.T * L_z.T * target
hess_lam_full = 2 * F.T * F

hess = sp.Matrix([
    [h00, hess_theta_full[0], hess_theta_full[1]],
    [hess_theta_full[0], hess_lam_full[0, 0] + C / (scales[0] ** 2), hess_lam_full[0, 1]],
    [hess_theta_full[1], hess_lam_full[1, 0], hess_lam_full[1, 1] + C / (scales[1] ** 2)]
])

print(to_ccode_matrix(hess))

{
    {l_x*(-1.2078711764132604*y[12] + 6.3914534971850268*y[14] + 6.2762827396372725*y[5]) + l_y*(1.2078711764132604*y[12] + 6.3914534971850268*y[14] - 6.2762827396372725*y[5]),-3.1381413698186362*y[1] - 1.5978633742962567*y[6] + 0.60393558820663018*y[8],3.1381413698186362*y[1] - 1.5978633742962567*y[6] - 0.60393558820663018*y[8]},
    {-3.1381413698186362*y[1] - 1.5978633742962567*y[6] + 0.60393558820663018*y[8],C/pow(l_x, 2) + 2.7925268031909272,0.23935944027350806},
    {3.1381413698186362*y[1] - 1.5978633742962567*y[6] - 0.60393558820663018*y[8],0.23935944027350806,C/pow(l_y, 2) + 2.7925268031909272}
};
